In [1]:
partition = 478

In [2]:
import sys
from train import main
from itertools import product  
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:
import re

def load_tested_configs(log_path):
    tested = set()
    with open(log_path, 'r') as f:
        for line in f:
            if line.startswith("Running:"):
                match = re.findall(r"[-\w.]+=\S+", line)
                if match:
                    # Normalize values to correct types
                    config = tuple([
                        int(re.search(r"=(\d+)", match[0]).group(1)),       # n_tree
                        int(re.search(r"=(\d+)", match[1]).group(1)),       # t_depth
                        int(re.search(r"=(\d+)", match[2]).group(1)),       # hd
                        int(re.search(r"=(\d+)", match[3]).group(1)),       # batch_size
                        float(re.search(r"=(\d+\.?\d*)", match[4]).group(1)), # feature_rate
                        float(re.search(r"=(\d+\.?\d*)", match[5]).group(1)), # dropout
                        float(re.search(r"=(\d+\.?\d*)", match[6]).group(1)), # lr
                    ])
                    tested.add(config)
    return tested


In [4]:
import random
from itertools import product
import sys

log_path = f"logs{partition}.txt"
tested_configs = load_tested_configs(log_path)

n_tree_values = [5, 10, 20, 50, 100]
tree_depth_values = [8, 9, 10, 11,12,13]
hidden_dim = [1024, 768]
batch_size_values = [128, 256, 512]
tree_feature_rates = [0.0, 0.1, 0.2, 0.3, 0.4]
feat_dropouts = [0.0, 0.1, 0.2, 0.3]
lrs = [0.001, 0.01]

n_iter = 50
best_score = 0
best_config = {}

param_space = list(product(
    n_tree_values,
    tree_depth_values,
    hidden_dim,
    batch_size_values,
    tree_feature_rates,
    feat_dropouts,
    lrs
))

best_acc = 0

available_configs = [cfg for cfg in param_space if cfg not in tested_configs]
sampled_configs = random.sample(available_configs, min(n_iter, len(available_configs)))
i = 1

for n_tree, t_depth, hd, batch_size, feature_rate, dropout, lr in sampled_configs:
    log_line = f"Running: n_tree={n_tree}, t_depth={t_depth}, hd={hd}, batch_size={batch_size}, feature_rate={feature_rate}, dropout={dropout}, lr={lr}"
    print(f"\n{log_line}")
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{log_line}\n")
    
    sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(n_tree),
        '-tree_depth', str(t_depth),
        '-batch_size', str(batch_size),
        '-hidden_dim', str(hd),
        '-tree_feature_rate', str(feature_rate),
        '-feat_dropout', str(dropout),
        '-lr', str(lr),
        '-epochs', '400',
        '-verbose', '0',
        '-jointly_training',
        '-searching', '1'
    ]

    print(f"{i} / 100")
    acc = main()
    #print(acc)
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{acc}\n")
    i =i + 1

    if acc > best_acc:
        best_acc = acc
        best_config = {
            'n_tree': n_tree,
            'tree_depth': t_depth,
            'batch_size': batch_size,
            'hidden_dim': hd,
            'tree_feature_rate': feature_rate,
            'feat_dropout': dropout,
            'lr': lr
        }

print("\nBest hyperparameter configuration:")
print(best_config)
print(f"Best accuracy: {best_acc}")



Running: n_tree=50, t_depth=9, hd=1024, batch_size=512, feature_rate=0.1, dropout=0.0, lr=0.01
1 / 100
Use gtd478 dataset


Patience: 100


Training Epochs:  79%|███████▉  | 316/400 [05:32<01:28,  1.05s/it]

Early stopping at epoch 317

Best Accuracy: 0.914671

Running: n_tree=20, t_depth=12, hd=768, batch_size=256, feature_rate=0.2, dropout=0.1, lr=0.001
2 / 100
Use gtd478 dataset


Patience: 100


...

Patience: 100


Training Epochs: 100%|██████████| 400/400 [23:38<00:00,  3.55s/it]



Best Accuracy: 0.917665

Running: n_tree=10, t_depth=13, hd=1024, batch_size=256, feature_rate=0.2, dropout=0.2, lr=0.001
50 / 100
Use gtd478 dataset
Patience: 100


Training Epochs: 100%|██████████| 400/400 [03:38<00:00,  1.83it/s]


Best Accuracy: 0.886228

Best hyperparameter configuration:
{'n_tree': 50, 'tree_depth': 13, 'batch_size': 512, 'hidden_dim': 768, 'tree_feature_rate': 0.1, 'feat_dropout': 0.2, 'lr': 0.01}
Best accuracy: 0.9276447105788423


In [5]:
#Running: n_tree=100, t_depth=11, hd=768, batch_size=512, feature_rate=0.3, dropout=0.1, lr=0.01
# 0.9067

# {'n_tree': 100, 'tree_depth': 11, 'batch_size': 512, 'hidden_dim': 768, 'tree_feature_rate': 0.3, 'feat_dropout': 0.1, 'lr': 0.01}
#0.9174


In [6]:
"""


========== Final Test Evaluation ==========
Model Parameters:
  Dataset: gtd478
  Hidden Dim: 768
  n_tree: 50, tree_depth: 13, tree_feature_rate: 0.1
  Batch size: 512, Dropout: 0.2, LR: 0.01

Best Accuracy: 0.9289
Weighted Precision: 0.9294, Recall: 0.9289, F1 Score: 0.9288, ROCAUC: 0.9976
Macro Precision: 0.9294, Recall: 0.9289, F1 Score: 0.9288, ROCAUC: 0.9976
Micro Precision: 0.9289, Recall: 0.9289, F1 Score: 0.9289, ROCAUC: 0.9985

"""

'\n\n========== Final Test Evaluation ==========\nModel Parameters:\n  Dataset: gtd478\n  Hidden Dim: 768\n  n_tree: 100, tree_depth: 12, tree_feature_rate: 0.3\n  Batch size: 512, Dropout: 0.1, LR: 0.01\n\nBest Accuracy: 0.9174\nWeighted Precision: 0.9178, Recall: 0.9174, F1 Score: 0.9169, ROCAUC: 0.9973\nMacro Precision: 0.9178, Recall: 0.9174, F1 Score: 0.9169, ROCAUC: 0.9973\nMicro Precision: 0.9174, Recall: 0.9174, F1 Score: 0.9174, ROCAUC: 0.9983\n'

In [7]:
sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(best_config['n_tree']),
        '-tree_depth', str(best_config['tree_depth']),
        '-batch_size', str(best_config['batch_size']),
        '-hidden_dim', str(best_config['hidden_dim']),
        '-epochs', '1500',
        '-verbose', '0',
        '-tree_feature_rate', str(best_config['tree_feature_rate']),
        '-feat_dropout', str(best_config['feat_dropout']),
        '-lr', str(best_config['lr']),
        '-jointly_training',
        '-searching', '0'
    ]

best_model, preds, targets, labels, epoch_logs = main()

Use gtd478 dataset


Patience: 300


Training Epochs:   3%|▎         | 50/1500 [01:37<44:42,  1.85s/it]

[Epoch 50] Train Loss: 0.3739, Eval Loss: 0.4081, Eval Accuracy: 0.9052


Training Epochs:   7%|▋         | 100/1500 [03:10<44:08,  1.89s/it]

...

[Epoch 900] Train Loss: 0.2543, Eval Loss: 0.3281, Eval Accuracy: 0.9296


Training Epochs:  63%|██████▎   | 950/1500 [29:24<16:51,  1.84s/it]

[Epoch 950] Train Loss: 0.2534, Eval Loss: 0.3231, Eval Accuracy: 0.9356


Training Epochs:  65%|██████▌   | 975/1500 [30:12<16:15,  1.86s/it]

Early stopping at epoch 976
Evaluating on test set with best model...


In [8]:
from sklearn.metrics import classification_report

print(classification_report(targets, preds))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       0.97      0.97      0.97       144
        African National Congress (South Africa)       1.00      1.00      1.00       144
                                Al-Qaida in Iraq       0.79      0.84      0.81       144
        Al-Qaida in the Arabian Peninsula (AQAP)       0.90      0.89      0.90       144
                                      Al-Shabaab       0.98      1.00      0.99       144
             Basque Fatherland and Freedom (ETA)       1.00      0.99      1.00       144
                                      Boko Haram       0.96      0.94      0.95       144
  Communist Party of India - Maoist (CPI-Maoist)       0.89      0.90      0.90       144
       Corsican National Liberation Front (FLNC)       0.99      1.00      0.99       144
                       Donetsk People's Republic       1.00      1.00      1.00       144
Farabundo

In [9]:
def plot_confusion_matrix(y_true, y_pred, labels, partition):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix (Partition gtd{partition})", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    save_path = f"results/confusion_matrix_partition_gtd{partition}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition gtd{partition} to {save_path}")



In [10]:
plot_confusion_matrix(targets, preds, labels, partition)

ValueError: At least one label specified must be in y_true